In [ ]:
import sys
import platform
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU NOT FOUND")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
!pip install -q "timesfm[torch]==2.0.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
import timesfm
import torch

print("TimesFM import: OK")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")^

TimesFM import: OK
CUDA available: True
GPU: Tesla T4


In [ ]:
import numpy as np
import timesfm
import torch

torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM 2.5 loaded and compiled successfully.")

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

TimesFM 2.5 loaded and compiled successfully.


In [ ]:
import pandas as pd

DATA_PATH = "/content/metr-la.h5"
SENSOR_ID = "773062"

# METR-LA dosyasını aç
df = pd.read_hdf(DATA_PATH)

print("Dataset shape:", df.shape)
print("Start time:", df.index.min())
print("End time:", df.index.max())
print("Number of sensors:", len(df.columns))
print("Sensor exists:", SENSOR_ID in df.columns)

# Ana sensörümüzün zaman serisini çıkar
series = df[SENSOR_ID].astype(float)

print("\nSensor:", SENSOR_ID)
print("Number of observations:", len(series))
print("First 10 values:")
print(series.head(10))

FileNotFoundError: File /content/metr-la.h5 does not exist

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving metr-la.h5 to metr-la.h5


In [ ]:
import os

print(os.path.exists("/content/metr-la.h5"))
print(os.path.getsize("/content/metr-la.h5"))

True
57038056


In [ ]:
import pandas as pd

DATA_PATH = "/content/metr-la.h5"
SENSOR_ID = "773062"

df = pd.read_hdf(DATA_PATH)

print("Dataset shape:", df.shape)
print("Start time:", df.index.min())
print("End time:", df.index.max())
print("Number of sensors:", len(df.columns))
print("Sensor exists:", SENSOR_ID in df.columns)

series = df[SENSOR_ID].astype(float)

print("\nSensor:", SENSOR_ID)
print("Number of observations:", len(series))
print("First 10 values:")
print(series.head(10))

Dataset shape: (34272, 207)
Start time: 2012-03-01 00:00:00
End time: 2012-06-27 23:55:00
Number of sensors: 207
Sensor exists: True

Sensor: 773062
Number of observations: 34272
First 10 values:


AttributeError: 'numpy.bytes_' object has no attribute 'freqstr'

In [ ]:
df.index.freq = None

series = df[SENSOR_ID].astype(float)

print("Sensor:", SENSOR_ID)
print("Number of observations:", len(series))
print("First 10 values:")
print(series.head(10))

Sensor: 773062
Number of observations: 34272
First 10 values:
2012-03-01 00:00:00    65.125000
2012-03-01 00:05:00    65.000000
2012-03-01 00:10:00    64.500000
2012-03-01 00:15:00     0.000000
2012-03-01 00:20:00     0.000000
2012-03-01 00:25:00    60.666667
2012-03-01 00:30:00    65.125000
2012-03-01 00:35:00    64.625000
2012-03-01 00:40:00    65.125000
2012-03-01 00:45:00    64.875000
Name: 773062, dtype: float64


In [ ]:
import numpy as np

CONTEXT_LENGTH = 96
FORECAST_HORIZON = 24

# Son %20'yi test bölgesi olarak ayır
test_start = int(len(series) * 0.80)

window_start = None

# Model sonucuna bakmadan ilk temiz 96+24 pencereyi bul
for start in range(test_start, len(series) - CONTEXT_LENGTH - FORECAST_HORIZON + 1):
    window = series.iloc[start:start + CONTEXT_LENGTH + FORECAST_HORIZON]

    if window.notna().all() and (window > 0).all():
        window_start = start
        break

assert window_start is not None, "Clean window could not be found."

context = series.iloc[
    window_start:
    window_start + CONTEXT_LENGTH
]

ground_truth = series.iloc[
    window_start + CONTEXT_LENGTH:
    window_start + CONTEXT_LENGTH + FORECAST_HORIZON
]

print("Clean window found ✅")
print("Context length:", len(context))
print("Forecast horizon:", len(ground_truth))

print("\nContext:")
print(context.index[0], "→", context.index[-1])

print("\nGround Truth:")
print(ground_truth.index[0], "→", ground_truth.index[-1])

print("\nAny zero in context?:", (context == 0).any())
print("Any zero in ground truth?:", (ground_truth == 0).any())

print("\nFirst 5 context values:")
print(context.head())

print("\nFirst 5 future values:")
print(ground_truth.head())

Clean window found ✅
Context length: 96
Forecast horizon: 24

Context:
2012-06-04 13:30:00 → 2012-06-04 21:25:00

Ground Truth:
2012-06-04 21:30:00 → 2012-06-04 23:25:00

Any zero in context?: False
Any zero in ground truth?: False

First 5 context values:
2012-06-04 13:30:00    59.375000
2012-06-04 13:35:00    57.875000
2012-06-04 13:40:00    60.888889
2012-06-04 13:45:00    51.750000
2012-06-04 13:50:00    59.285714
Name: 773062, dtype: float64

First 5 future values:
2012-06-04 21:30:00    65.125
2012-06-04 21:35:00    56.625
2012-06-04 21:40:00    64.000
2012-06-04 21:45:00    64.125
2012-06-04 21:50:00    58.000
Name: 773062, dtype: float64


In [ ]:
import numpy as np
import time

# TimesFM'e yalnızca geçmiş 96 ölçümü veriyoruz.
timesfm_input = context.to_numpy(dtype=np.float32)

start_time = time.perf_counter()

point_forecast, quantile_forecast = model.forecast(
    horizon=FORECAST_HORIZON,
    inputs=[timesfm_input]
)

inference_time = time.perf_counter() - start_time

# Tek bir time series kullandığımız için batch'in ilk elemanını alıyoruz.
timesfm_prediction = point_forecast[0]

print("TimesFM inference completed ✅")
print("Input shape:", timesfm_input.shape)
print("Prediction shape:", timesfm_prediction.shape)
print(f"Inference time: {inference_time:.4f} seconds")

print("\nFirst 5 TimesFM predictions:")
print(timesfm_prediction[:5])

print("\nFirst 5 Ground Truth values:")
print(ground_truth.to_numpy()[:5])

TimesFM inference completed ✅
Input shape: (96,)
Prediction shape: (24,)
Inference time: 1.0109 seconds

First 5 TimesFM predictions:
[64.80135 64.7375  64.30818 64.16092 63.87301]

First 5 Ground Truth values:
[65.125 56.625 64.    64.125 58.   ]


In [ ]:
import numpy as np

y_true = ground_truth.to_numpy(dtype=np.float32)
y_pred = np.asarray(timesfm_prediction, dtype=np.float32)

assert len(y_true) == len(y_pred) == FORECAST_HORIZON

timesfm_mae = np.mean(np.abs(y_true - y_pred))
timesfm_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

print("TimesFM Results")
print("-----------------")
print(f"MAE : {timesfm_mae:.4f}")
print(f"RMSE: {timesfm_rmse:.4f}")

TimesFM Results
-----------------
MAE : 2.3031
RMSE: 2.9794


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "timestamp": ground_truth.index,
    "ground_truth": y_true,
    "timesfm_prediction": y_pred
})

results_df.to_csv("/content/timesfm_predictions.csv", index=False)

print(results_df.head())
print("\nSaved: /content/timesfm_predictions.csv")0 2012-06-04 21:30:00        65.125           64.801353
1 2012-06-04 21:35:00        56.625           64.737503
2 2012-06-04 21:40:00        64.000           64.308182
3 2012-06-04 21:45:00        64.125           64.160919
4 2012-06-04 21:50:00        58.000           63.873009

Saved: /content/timesfm_predictions.csv$0

            timestamp  ground_truth  timesfm_prediction
0 2012-06-04 21:30:00        65.125           64.801353
1 2012-06-04 21:35:00        56.625           64.737503
2 2012-06-04 21:40:00        64.000           64.308182
3 2012-06-04 21:45:00        64.125           64.160919
4 2012-06-04 21:50:00        58.000           63.873009

Saved: /content/timesfm_predictions.csv


In [ ]:
eval_window = pd.DataFrame({
    "timestamp": list(context.index) + list(ground_truth.index),
    "traffic_speed": list(context.values) + list(ground_truth.values),
    "split": ["context"] * len(context) + ["ground_truth"] * len(ground_truth)
})

eval_window.to_csv("/content/evaluation_window.csv", index=False)

print(eval_window.head())
print("\nRows:", len(eval_window))
print("Context rows:", (eval_window["split"] == "context").sum())
print("Ground truth rows:", (eval_window["split"] == "ground_truth").sum())

            timestamp  traffic_speed    split
0 2012-06-04 13:30:00      59.375000  context
1 2012-06-04 13:35:00      57.875000  context
2 2012-06-04 13:40:00      60.888889  context
3 2012-06-04 13:45:00      51.750000  context
4 2012-06-04 13:50:00      59.285714  context

Rows: 120
Context rows: 96
Ground truth rows: 24


In [ ]:
from google.colab import files

files.download("/content/evaluation_window.csv")
files.download("/content/timesfm_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving evaluation_window.csv to evaluation_window.csv
Saving timesfm_predictions.csv to timesfm_predictions.csv


In [ ]:
!ls -lh /content/evaluation_window.csv /content/timesfm_predictions.csv

-rw-r--r-- 1 root root 4.6K Aug 14 14:57 /content/evaluation_window.csv
-rw-r--r-- 1 root root  906 Aug 14 14:57 /content/timesfm_predictions.csv


In [ ]:
import numpy as np
import pandas as pd
import torch
import timesfm

# ============================================================
# LOCKED AUDIT CONFIG
# ============================================================

MODEL_ID = "google/timesfm-2.5-200m-pytorch"
EVAL_PATH = "/content/evaluation_window.csv"
OLD_RESULT_PATH = "/content/timesfm_predictions.csv"

CONTEXT_LENGTH = 96
FORECAST_HORIZON = 24

# ============================================================
# 1. VERIFY EXACT DATA
# ============================================================

ev = pd.read_csv(EVAL_PATH, parse_dates=["timestamp"])
old = pd.read_csv(OLD_RESULT_PATH, parse_dates=["timestamp"])

context_df = ev[ev["split"] == "context"].copy()
gt_df = ev[ev["split"] == "ground_truth"].copy()

assert len(context_df) == 96
assert len(gt_df) == 24
assert len(old) == 24

context = context_df["traffic_speed"].to_numpy(dtype=np.float32)
y_true = gt_df["traffic_speed"].to_numpy(dtype=np.float64)

# Saved TimesFM CSV must refer to EXACT same ground truth.
assert np.array_equal(
    old["timestamp"].to_numpy(),
    gt_df["timestamp"].to_numpy()
), "Timestamp mismatch!"

assert np.allclose(
    old["ground_truth"].to_numpy(dtype=float),
    y_true
), "Ground truth mismatch!"

print("Exact evaluation window: OK")

# ============================================================
# 2. LOAD CURRENT TIMESFM 2.5
# ============================================================

print("Loading:", MODEL_ID)

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    MODEL_ID
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM 2.5 model: OK")

# ============================================================
# 3. RE-RUN ZERO-SHOT FORECAST
# ============================================================

with torch.no_grad():
    point_forecast, quantile_forecast = model.forecast(
        horizon=FORECAST_HORIZON,
        inputs=[context],
    )

point = np.asarray(point_forecast[0], dtype=np.float64)
q50 = np.asarray(quantile_forecast[0, :, 5], dtype=np.float64)

saved = old["timesfm_prediction"].to_numpy(dtype=np.float64)

assert point.shape == (24,)
assert q50.shape == (24,)
assert saved.shape == (24,)

# ============================================================
# 4. AUDIT EQUALITY
# ============================================================

point_vs_q50_maxdiff = np.max(np.abs(point - q50))
point_vs_saved_maxdiff = np.max(np.abs(point - saved))

print()
print("======================================")
print("TIMESFM 2.5 POINT-FORECAST AUDIT")
print("======================================")

print("point forecast vs q=0.5 max abs diff:",
      point_vs_q50_maxdiff)

print("new point forecast vs saved CSV max abs diff:",
      point_vs_saved_maxdiff)

assert np.allclose(
    point,
    q50,
    rtol=1e-5,
    atol=1e-5
), "FAIL: point forecast != q0.5 median"

assert np.allclose(
    point,
    saved,
    rtol=1e-5,
    atol=1e-5
), "FAIL: saved TimesFM predictions differ from rerun"

print()
print("✅ point_forecast == q0.5 median")
print("✅ rerun forecast == saved timesfm_predictions.csv")

# ============================================================
# 5. RECOMPUTE METRICS
# ============================================================

mae = np.mean(np.abs(y_true - point))
rmse = np.sqrt(np.mean((y_true - point) ** 2))

print()
print("First 5 Ground Truth:")
print(y_true[:5])

print()
print("First 5 TimesFM q0.5:")
print(point[:5])

print()
print("====================")
print("VERIFIED METRICS")
print("====================")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

print()
print("✅ TIMESFM 2.5 AUDIT PASSED")

Exact evaluation window: OK
Loading: google/timesfm-2.5-200m-pytorch


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

TimesFM 2.5 model: OK

TIMESFM 2.5 POINT-FORECAST AUDIT
point forecast vs q=0.5 max abs diff: 0.0
new point forecast vs saved CSV max abs diff: 3.750000004743015e-06

✅ point_forecast == q0.5 median
✅ rerun forecast == saved timesfm_predictions.csv

First 5 Ground Truth:
[65.125 56.625 64.    64.125 58.   ]

First 5 TimesFM q0.5:
[64.80135345 64.73750305 64.30818176 64.16091919 63.87300873]

VERIFIED METRICS
MAE : 2.3031
RMSE: 2.9794

✅ TIMESFM 2.5 AUDIT PASSED


In [ ]:
!pip install -q "timesfm[torch]==2.0.2"

import timesfm
import importlib.metadata as metadata

print("TimesFM package:", metadata.version("timesfm"))
print("TimesFM import: OK")

TimesFM package: 2.0.2
TimesFM import: OK


In [ ]:
!nvidia-smi

Fri Aug 14 15:33:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving evaluation_window_sensor_717608.csv to evaluation_window_sensor_717608.csv


In [ ]:
!ls -lh /content/evaluation_window_sensor_717608.csv

-rw-r--r-- 1 root root 4.6K Aug 14 15:34 /content/evaluation_window_sensor_717608.csv


In [ ]:
import os
import pandas as pd
import numpy as np

PATH = "/content/evaluation_window_sensor_717608.csv"

assert os.path.exists(PATH)

df = pd.read_csv(PATH, parse_dates=["timestamp"])

context = df[df["split"] == "context"]
gt = df[df["split"] == "ground_truth"]

assert len(df) == 120
assert len(context) == 96
assert len(gt) == 24
assert context["traffic_speed"].notna().all()
assert gt["traffic_speed"].notna().all()
assert np.isfinite(context["traffic_speed"]).all()
assert np.isfinite(gt["traffic_speed"]).all()
assert (context["traffic_speed"] > 0).all()
assert (gt["traffic_speed"] > 0).all()

print("Sensor B file:", PATH)
print("Context:", len(context))
print("Horizon:", len(gt))
print("First 5 context:", context["traffic_speed"].to_numpy()[:5])
print("First 5 GT:", gt["traffic_speed"].to_numpy()[:5])
print("✅ SENSOR 717608 INPUT VERIFIED")

Sensor B file: /content/evaluation_window_sensor_717608.csv
Context: 96
Horizon: 24
First 5 context: [66.25       66.375      66.88888889 66.5        67.42857143]
First 5 GT: [67.625      68.25       68.44444444 68.625      68.66666667]
✅ SENSOR 717608 INPUT VERIFIED


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import timesfm

# ============================================================
# SENSOR B — TIMESFM 2.5 ZERO-SHOT
# Same configuration verified on Sensor A
# ============================================================

MODEL_ID = "google/timesfm-2.5-200m-pytorch"
INPUT_PATH = "/content/evaluation_window_sensor_717608.csv"
OUTPUT_PATH = "/content/timesfm_predictions_sensor_717608.csv"

SENSOR_ID = "717608"
CONTEXT_LENGTH = 96
FORECAST_HORIZON = 24

# ============================================================
# 1. VERIFY GPU + DATA
# ============================================================

assert torch.cuda.is_available(), "CUDA is not available."
print("GPU:", torch.cuda.get_device_name(0))

df = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])

context_df = df[df["split"] == "context"].copy()
gt_df = df[df["split"] == "ground_truth"].copy()

assert len(context_df) == CONTEXT_LENGTH
assert len(gt_df) == FORECAST_HORIZON

context = context_df["traffic_speed"].to_numpy(dtype=np.float32)
y_true = gt_df["traffic_speed"].to_numpy(dtype=np.float64)

assert context.shape == (96,)
assert y_true.shape == (24,)
assert np.isfinite(context).all()
assert np.isfinite(y_true).all()

print("Input validation: OK")
print("Context points:", len(context))
print("Ground-truth points:", len(y_true))

# ============================================================
# 2. LOAD TIMESFM 2.5
# ============================================================

print("\nLoading:", MODEL_ID)

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    MODEL_ID
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM 2.5 model: OK")

# ============================================================
# 3. ZERO-SHOT INFERENCE
# ============================================================

start = time.perf_counter()

with torch.no_grad():
    point_forecast, quantile_forecast = model.forecast(
        horizon=FORECAST_HORIZON,
        inputs=[context],
    )

elapsed = time.perf_counter() - start

point = np.asarray(point_forecast[0], dtype=np.float64)
quantiles = np.asarray(quantile_forecast[0], dtype=np.float64)

assert point.shape == (24,)
assert quantiles.shape[0] == 24

# TimesFM q=0.5
q50 = quantiles[:, 5]

# ============================================================
# 4. STRICT OUTPUT AUDIT
# ============================================================

assert np.isfinite(point).all(), \
    "FAIL: TimesFM point forecast contains NaN/Inf."

assert np.isfinite(q50).all(), \
    "FAIL: TimesFM q0.5 contains NaN/Inf."

point_q50_maxdiff = np.max(
    np.abs(point - q50)
)

assert np.allclose(
    point,
    q50,
    atol=1e-5,
    rtol=1e-5
), f"FAIL: point forecast != q0.5, max diff={point_q50_maxdiff}"

prediction = q50.copy()

print("\nTimesFM output validation: PASSED")
print("Prediction shape:", prediction.shape)
print("NaN count:", int(np.isnan(prediction).sum()))
print("Inf count:", int(np.isinf(prediction).sum()))
print("point vs q0.5 max abs diff:", point_q50_maxdiff)

# ============================================================
# 5. METRICS
# ============================================================

mae = np.mean(
    np.abs(y_true - prediction)
)

rmse = np.sqrt(
    np.mean((y_true - prediction) ** 2)
)

assert np.isfinite(mae)
assert np.isfinite(rmse)

# ============================================================
# 6. SAVE RESULT
# ============================================================

result = pd.DataFrame({
    "timestamp": gt_df["timestamp"].to_numpy(),
    "ground_truth": y_true,
    "timesfm_prediction": prediction,
})

result.to_csv(
    OUTPUT_PATH,
    index=False
)

# ============================================================
# 7. REPORT
# ============================================================

print("\n======================================")
print("SENSOR 717608 — TIMESFM 2.5 RESULT")
print("======================================")
print("Model:", MODEL_ID)
print("Sensor:", SENSOR_ID)
print("Context:", CONTEXT_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Training / fine-tuning: NONE")
print("Point forecast: q=0.5 median")

print("\nFirst 5 Ground Truth:")
print(y_true[:5])

print("\nFirst 5 TimesFM Predictions:")
print(prediction[:5])

print("\n====================")
print("METRICS")
print("====================")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

print(f"\nInference time (T4): {elapsed:.4f} sec")

print("\nSaved:", OUTPUT_PATH)
print("Rows:", len(result))

print("\n✅ SENSOR 717608 TIMESFM 2.5 AUDIT PASSED")

GPU: Tesla T4
Input validation: OK
Context points: 96
Ground-truth points: 24

Loading: google/timesfm-2.5-200m-pytorch


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

TimesFM 2.5 model: OK

TimesFM output validation: PASSED
Prediction shape: (24,)
NaN count: 0
Inf count: 0
point vs q0.5 max abs diff: 0.0

SENSOR 717608 — TIMESFM 2.5 RESULT
Model: google/timesfm-2.5-200m-pytorch
Sensor: 717608
Context: 96
Forecast horizon: 24
Training / fine-tuning: NONE
Point forecast: q=0.5 median

First 5 Ground Truth:
[67.625      68.25       68.44444444 68.625      68.66666667]

First 5 TimesFM Predictions:
[68.4573822  68.4526825  68.42696381 68.48982239 68.51974487]

METRICS
MAE : 1.3665
RMSE: 2.5878

Inference time (T4): 0.9131 sec

Saved: /content/timesfm_predictions_sensor_717608.csv
Rows: 24

✅ SENSOR 717608 TIMESFM 2.5 AUDIT PASSED


In [ ]:
from google.colab import files

files.download("/content/timesfm_predictions_sensor_717608.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q "timesfm[torch]==2.0.2"

import torch
import timesfm
import importlib.metadata as metadata

print("TimesFM package:", metadata.version("timesfm"))
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("✅ TIMESFM + T4 READY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
TimesFM package: 2.0.2
CUDA available: True
GPU: Tesla T4
✅ TIMESFM + T4 READY
